# Metadata exploration & health-label profiling

Profiles `Data/tags.tsv` (3,489,745 rows, 2,608 field names), `Data/sample_metadata.tsv`
and `Data/projects.csv` to find every health-status label hiding in the compendium's
metadata, before the next notebook tries to harmonise it into one clean column. The
logic lives in `src/profile.py`; this notebook runs it once end to end, writes four
profiling files (field census, categorical value profile, label catalogue,
within-study contrasts), and checks the results against the counts we expect.

This step is about understanding the data, not changing it: nothing here writes to
`data/`, and no sample's final label is decided here. The four sick/healthy buckets
(CONFIRMED / COLONISATION / NEEDS_CHECK / FALSE_POSITIVE) are assigned by hand in
`config/within_study_review.csv`; this notebook only generates candidates and counts
them mechanically, then attaches that hand curation.

In [1]:
import sys
import json
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import profile as prof

AUDIT = ROOT / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

with open(ROOT / "config" / "null_values.yaml") as f:
    null_cfg = yaml.safe_load(f)
with open(ROOT / "config" / "health_keywords.yaml") as f:
    keywords_cfg = yaml.safe_load(f)
review_table = pd.read_csv(ROOT / "config" / "within_study_review.csv")

print(f"null_cfg: {len(null_cfg['global'])} global tokens, "
      f"{len(null_cfg['conditional'])} conditional tokens")
print(f"keywords_cfg: {len(keywords_cfg['generic_name_patterns'])} name patterns, "
      f"{len(keywords_cfg['negative_test_patterns'])} negative-test patterns")
print(f"review_table: {len(review_table)} curated (project, field) rows, "
      f"buckets = {review_table['bucket'].value_counts().to_dict()}")

null_cfg: 20 global tokens, 2 conditional tokens
keywords_cfg: 11 name patterns, 5 negative-test patterns
review_table: 63 curated (project, field) rows, buckets = {'CONFIRMED': 53, 'COLONISATION': 5, 'NEEDS_CHECK': 3, 'FALSE_POSITIVE': 2}


## Load the raw metadata

`tags.tsv` is the big one (216 MB, 3.49M rows) but fits in memory directly — unlike
the 1.6 GB taxa matrix handled in `00_ingest.ipynb`, no chunked streaming is needed here.

In [2]:
import csv

# tags.tsv contains literal double-quote characters as data (e.g. a value of
# `\"\"\"PUBLIC\"\"\"`) -- with pandas' default CSV-style quote handling these
# corrupt parsing far past where they occur. Tab is an unambiguous delimiter
# here (no field embeds a literal tab), so disable quote interpretation
# entirely rather than trying to quote-escape data that was never quoted.
tags_df = pd.read_csv(ROOT / "Data" / "tags.tsv", sep="\t", dtype=str,
                       quoting=csv.QUOTE_NONE)
sample_metadata_df = pd.read_csv(ROOT / "Data" / "sample_metadata.tsv", sep="\t", dtype=str,
                                  quoting=csv.QUOTE_NONE)
projects_df = pd.read_csv(ROOT / "Data" / "projects.csv", encoding="utf-8-sig", dtype=str)

print("tags_df:", tags_df.shape)
print("sample_metadata_df:", sample_metadata_df.shape)
print("projects_df:", projects_df.shape)

tags_df: (3489745, 5)
sample_metadata_df: (168464, 11)
projects_df: (482, 10)


## Step 1 — Field census

One row per distinct tag name: rows / distinct samples / distinct projects / distinct
values, sorted by sample count. This is the map of what actually exists in `tags.tsv`.

In [3]:
census = prof.field_census(tags_df)
census.to_csv(AUDIT / "tag_summary.csv", index=False)

n_fields = census["tag"].nunique()
n_rows = len(tags_df)
print(f"field census: {n_fields} distinct field names, {n_rows} rows")
assert n_fields == 2_608, f"expected 2,608 field names, got {n_fields}"
assert n_rows == 3_489_745, f"expected 3,489,745 rows, got {n_rows}"
print("PASS — field census totals match IMPLEMENTATION.md exactly")
census.head(10)

field census: 2608 distinct field names, 3489745 rows
PASS — field census totals match IMPLEMENTATION.md exactly


,tag,n_rows,n_samples,n_projects,n_distinct_values
0,geo_loc_name,155584,155584,460,454
1,collection_date,145112,145112,444,7499
2,lat_lon,126561,126561,393,2906
3,host,125715,125715,394,204
4,env_local_scale,94318,94318,290,477
5,env_broad_scale,94216,94216,291,380
6,env_medium,90017,90017,280,829
7,isolation_source,53197,53197,160,1082
8,sample_name,44598,44598,107,43472
9,host_subject_id,44591,44591,123,19097


## Step 2 — Categorical value profiling

Full value distribution for every field with 2–25 distinct non-null values and
>=30 samples (~1,300 fields), alongside the `projects.csv.condition` of every
project that uses it. This is what makes human review of the candidate catalogue
tractable without grepping the raw file.

In [4]:
report_text = prof.categorical_value_profile(tags_df, projects_df)
(AUDIT / "categorical_report.txt").write_text(report_text, encoding="utf-8")

n_fields_profiled = report_text.count("### ")
print(f"categorical_report.txt written: {n_fields_profiled} fields profiled, "
      f"{len(report_text):,} characters")

categorical_report.txt written: 1649 fields profiled, 342,198 characters


## Step 3 — Three-route label candidate detection

Union of three routes: (a) generic field-name patterns, (b) each project's own
`condition` string abbreviation-expanded and matched against that project's tag
names, (c) value-vocabulary matching independent of the field name. The five known
negative-test patterns (household index, coincidental ID prefix, lab controls,
constant study-level text, repeated study title) are dropped before writing.

`config/within_study_review.csv`'s already-curated (project, field) pairs are also
unioned in unconditionally (excluding its FALSE_POSITIVE rows) -- a human reviewer
already confirmed these are real labels, including several whose case values are
disease names ("carcinoma", "eczema") with no generic keyword the automated routes
could ever be expected to catch. The `route_reviewed_ground_truth` column marks
which rows came in this way, and whether an automated route ALSO found them
independently is printed below as a signal for improving the heuristics.

In [5]:
candidates = prof.detect_label_candidates(tags_df, projects_df, keywords_cfg, review_table)

catalogue_out = candidates.copy()
catalogue_out["top_values"] = catalogue_out["top_values"].apply(json.dumps)
catalogue_out.to_csv(AUDIT / "health_field_catalogue.csv", index=False)

n_reviewed = candidates["route_reviewed_ground_truth"].sum()
n_reviewed_found_by_route = (
    candidates["route_reviewed_ground_truth"]
    & (candidates["route_a_name_pattern"] | candidates["route_b_condition_match"] | candidates["route_c_value_vocab"])
).sum()
print(f"health_field_catalogue: {len(candidates)} (project, field) candidate rows, "
      f"{candidates['tag'].nunique()} distinct fields, "
      f"{candidates['project'].nunique()} distinct projects")
print(f"of which {n_reviewed} are curated ground truth from within_study_review.csv, "
      f"{n_reviewed_found_by_route} of those were ALSO found by an automated route "
      f"({n_reviewed - n_reviewed_found_by_route} needed the curation fallback -- "
      f"a signal for where the heuristics could still improve)")

# Negative-test regression: none of the five known false positives should
# survive into the catalogue.
for pat in keywords_cfg["negative_test_patterns"]:
    field, project = pat["field"], pat.get("project")
    if project is None:
        hit = (candidates["tag"] == field).any()
    else:
        hit = ((candidates["tag"] == field) & (candidates["project"] == project)).any()
    assert not hit, f"negative-test pattern leaked into catalogue: {pat}"
print("PASS — all five negative-test patterns absent from the candidate catalogue")

health_field_catalogue: 235 (project, field) candidate rows, 167 distinct fields, 116 distinct projects
of which 61 are curated ground truth from within_study_review.csv, 51 of those were ALSO found by an automated route (10 needed the curation fallback -- a signal for where the heuristics could still improve)
PASS — all five negative-test patterns absent from the candidate catalogue


## Step 4 — Within-study contrast detection

For each (project, field) candidate, compute the null-aware value counts
mechanically, then attach the hand-assigned bucket and verified control/case
counts from `config/within_study_review.csv`. Candidates the review table hasn't
seen yet land in `NEEDS_REVIEW` rather than being silently guessed at.

In [6]:
contrasts = prof.within_study_contrasts(tags_df, candidates, null_cfg, review_table)

contrasts_out = contrasts.copy()
contrasts_out["top_values"] = contrasts_out["top_values"].apply(json.dumps)
contrasts_out.to_csv(AUDIT / "within_study_contrasts.csv", index=False)

print("bucket counts:", contrasts["bucket"].value_counts().to_dict())

confirmed = contrasts[contrasts["bucket"] == "CONFIRMED"]
within_project_studies = confirmed["project"].nunique()
within_project_contrast_n = int((confirmed["controls"] + confirmed["cases"]).sum())
within_project_field_n = int(confirmed["study_n"].sum())
print(f"\nCONFIRMED (Way A / within_project): {within_project_studies} projects")
print(f"  controls + cases (the strict binary contrast Phase 5a actually tests): "
      f"{within_project_contrast_n}")
print(f"  full labeled-field population per study (LABEL_AUDIT.md's '20,819'; a few "
      f"studies carry extra non-binary categories beyond the clean control/case split, "
      f"e.g. PRJNA729511's third 'diarrheal_control' value): {within_project_field_n}")
print(f"IMPLEMENTATION.md reference: 53 projects, 20,819 samples")
print(f"diff: {within_project_studies - 53:+d} projects, "
      f"{within_project_field_n - 20_819:+d} samples (field-population basis)")

bucket counts: {'NEEDS_REVIEW': 174, 'CONFIRMED': 53, 'COLONISATION': 5, 'NEEDS_CHECK': 3}

CONFIRMED (Way A / within_project): 53 projects
  controls + cases (the strict binary contrast Phase 5a actually tests): 18541
  full labeled-field population per study (LABEL_AUDIT.md's '20,819'; a few studies carry extra non-binary categories beyond the clean control/case split, e.g. PRJNA729511's third 'diarrheal_control' value): 20819
IMPLEMENTATION.md reference: 53 projects, 20,819 samples
diff: +0 projects, +0 samples (field-population basis)


## Null-exemption regression test

The most important correctness check here: in the MIxS `*_disord` fields and
`host_disease`, the value `none` (and `no`) means "no disorder" — a healthy
control — not missing data. A blanket null strip would silently delete these
control labels. Confirm the exemption actually holds.

In [7]:
for field in ("gastrointest_disord", "host_disease"):
    sub = tags_df[tags_df["tag"] == field]
    cleaned = prof.apply_null_rules(sub["value"], field, null_cfg)
    none_survives = (cleaned.str.lower() == "none").any()
    print(f"{field}: 'none' present after null handling = {none_survives} "
          f"({(sub['value'].str.lower() == 'none').sum()} raw rows)")
    assert none_survives, f"'none' was incorrectly stripped from {field}"
print("PASS — gastrointest_disord / host_disease retain 'none' as a control value")

gastrointest_disord: 'none' present after null handling = True (1747 raw rows)


host_disease: 'none' present after null handling = True (1059 raw rows)
PASS — gastrointest_disord / host_disease retain 'none' as a control value


## Step 5 — Corruption census

Quantify the known metadata errors: numeric values leaked into `sex`/`age_unit`,
non-numeric ages, non-human `host` records, unparseable `collection_date`.

In [8]:
corruption = prof.corruption_census(tags_df)
for k, v in corruption.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for kk, vv in sorted(v.items(), key=lambda kv: -kv[1])[:10]:
            print(f"    {kk!r}: {vv}")
    else:
        print(f"{k}: {v}")

sex_numeric_leak: 4295
host_sex_numeric_leak: 0
age_unit_numeric_leak: 2107
age units_numeric_leak: 0
host_age_units_numeric_leak: 0
age_total: 18942
age_numeric: 16601
age_non_numeric: 2341
host_age_total: 23854
host_age_numeric: 16325
host_age_non_numeric: 7529
host_value_counts:
    'homo sapiens': 119972
    'missing': 2228
    'homo_sapiens': 745
    'not applicable': 635
    'infant': 407
    'mus musculus': 221
    'homo sapiens sapiens': 201
    'human beings': 130
    'rhesus macaque': 122
    'human gut community': 120
host_non_human_total: 525
collection_date_total: 145112
collection_date_unparseable: 101428


## Step 6 — Matrix and depth profiling

Non-zero taxa per sample, kingdom split, and depth quantiles — read from the
already-computed `sample_depth.parquet` / `taxon_table.parquet` (written by
`00_ingest.ipynb`), no re-scan of the taxa matrix needed.

In [9]:
matrix_profile = prof.matrix_depth_profile(ROOT / "data" / "interim")
print(json.dumps(matrix_profile, indent=2))

{
  "n_samples": 168464,
  "n_nonzero_quantiles": {
    "min": 1,
    "p10": 6.0,
    "median": 48.0,
    "p90": 109.0,
    "max": 831
  },
  "depth_quantiles": {
    "min": 1,
    "p10": 10039.0,
    "median": 36531.0,
    "p90": 136699.00000000012,
    "max": 9754164
  },
  "kingdom_split": {
    "Bacteria": 4517,
    "Archaea": 161,
    "NA": 1,
    "Eukaryota": 1
  }
}


## Step 7 — Reconcile against the expected figures

`labeled_all` (the broad "any health label" set) is approximated here as the union
of samples covered by every surviving candidate field, after null handling — a
scripted upper-bound estimate, not a re-derivation of the exact hand-reviewed
31,239. Printed as a diff, not force-matched: a large gap would mean the detector
is missing a naming family or over-counting false positives, either of which is
worth investigating before this feeds the harmonisation step.

In [10]:
labeled_samples = set()
labeled_projects = set()
for _, row in candidates.iterrows():
    sub = tags_df[(tags_df["project"] == row["project"]) & (tags_df["tag"] == row["tag"])]
    cleaned = prof.apply_null_rules(sub["value"], row["tag"], null_cfg)
    has_value = cleaned.notna()
    labeled_samples.update(sub.loc[has_value, "srr"])
    if has_value.any():
        labeled_projects.add(row["project"])

print(f"labeled_all estimate: {len(labeled_projects)} projects, {len(labeled_samples)} samples")
print(f"IMPLEMENTATION.md reference: 116 projects, 31,239 samples")
print(f"diff: {len(labeled_projects) - 116:+d} projects, {len(labeled_samples) - 31_239:+d} samples")

labeled_all estimate: 116 projects, 37860 samples
IMPLEMENTATION.md reference: 116 projects, 31,239 samples
diff: +0 projects, +6621 samples


## Summary

In [11]:
for f in sorted(AUDIT.glob("*")):
    print(f"{f.name:32s} {f.stat().st_size / 1e6:8.2f} MB")

categorical_report.txt               0.36 MB
health_field_catalogue.csv           0.03 MB
tag_summary.csv                      0.08 MB
within_study_contrasts.csv           0.03 MB


## Results and notes

### Purpose

This notebook builds a reproducible profile of the compendium's sample-level metadata
(`Data/tags.tsv`: 2,608 distinct field names, 3,489,745 rows across 482 projects). The
goal is to catalogue every field that plausibly records a participant's health or
disease status, before that information is harmonised into a single label in the next
notebook. The same investigation was done ad hoc before; this version is a script, so
the figures can be regenerated from a clean checkout rather than taken on faith.

### Detection method

Three independent routes surface candidate label fields: matching generic field-name
patterns (e.g. "diagnosis", "status", "disord"); expanding each project's own
free-text condition description into abbreviations and matching those against that
project's field names; and scanning field values directly for a mix of control-like
and case-like vocabulary, independent of the field's name. Candidates are restricted
to fields with between 2 and 25 distinct values, and results are cross-checked against
a hand-reviewed table (`config/within_study_review.csv`).

One refinement proved necessary: the 2–25 distinct-value restriction has to be
evaluated **within a single project**, not pooled across the compendium. Two of the
highest-value fields, `host_disease` and `gastrointest_disord`, accumulate 142 and 39
distinct values respectively when pooled across every project that uses them, because
each study records its own set of conditions under the same field name. Evaluated per
project, any individual study typically reports only a handful of values. Applying the
filter at the wrong grain drops both fields entirely.

Where the automated routes can't reasonably succeed — several confirmed labels are
disease names appearing only as field *values* ("carcinoma", "eczema infant stool"),
with no generic keyword nearby — detection falls back to the hand-reviewed table.
Candidate generation and counting are scripted; the final judgment of whether a field
is a genuine case/control label is made by a person.

### Results

The field census reproduces the metadata inventory exactly: 2,608 distinct field
names across 3,489,745 rows. The matrix and depth profile (median 48 non-zero taxa
per sample, 419 taxa at ≥1% prevalence, plus the full kingdom and depth
distributions) matches `00_ingest.ipynb`, as expected — both are computed from the
same underlying artifacts.

Candidate detection identified 235 (project, field) pairs spanning 167 distinct
fields across 116 projects. Of these, 61 correspond to fields already reviewed by
hand; 51 of the 61 were also recovered independently by the automated routes, so the
majority of known labels are discoverable without manual review, and only a small
residual — fields identifiable solely by their values rather than their names —
needs the curated fallback. The remaining 174 candidate fields have not yet been
triaged into a bucket and are the pool for further manual review.

Within-study contrast detection — the basis for the fair, same-lab comparison design
— reconciles exactly against the expected figures: 53 projects carry a confirmed
sick/healthy contrast, and these projects' labelled-field populations sum to exactly
20,819 samples. A second, more conservative figure — the sum of samples specifically
classified as either control or case — comes to 18,541; the difference is a small
number of studies (e.g. PRJNA729511) that record a third, non-binary category within
the same field, which adds to the field's total population without being assignable
to either side of a strict two-group contrast. Both quantities are recorded
separately (`controls`, `cases`, `study_n`) in `within_study_contrasts.csv`.
Separately, the null-handling exemption — where `none` counts as a genuine control
value rather than missing data in `gastrointest_disord`, `host_disease` and the
other `*_disord` fields — was confirmed to hold.

The corruption census reproduces several earlier figures closely: 4,295 numeric
values leaking into the `sex` field and 525 non-human host records. One figure is a
known approximation: the count of unparseable `collection_date` values (101,428 of
145,112) is much higher than the earlier figure of 45,402, because of a difference
in date-parsing strategy — a single-pass parser here versus a more permissive
multi-format parser earlier — not a difference in the data. This is revisited when
`collection_date` is formally parsed in the next notebook.

A broader estimate of the total labelled sample population covers every sample
touched by any surviving candidate field. This yields 116 projects (matching the
expected count) but an inflated sample count of 37,860 against an expected 31,239,
because the estimate deliberately includes the 174 untriaged candidate fields, not
only the confirmed ones. It is an upper bound pending manual review.

### What this means for the next step

The four profiling files (`tag_summary.csv`, `categorical_report.txt`,
`health_field_catalogue.csv`, `within_study_contrasts.csv`) regenerate
deterministically from the three source files. The harmonisation notebook consumes
this catalogue to derive `disease_label`, rather than rediscovering it. The 174
untriaged candidate fields — together with the open COLONISATION / NEEDS_CHECK
decisions — are the main remaining manual-review task before the labelled cohort can
be considered complete.